# Annotate an RO-Crate with structured metadata

This notebook enriches an existing RO-Crate in ROHub with three layers of metadata:

1. **Extracted from data** — read CF attributes from Zarr (variables, coverage, resolution)
2. **I-ADOPT variable decomposition** — decompose variable names into Property + ObjectOfInterest + Matrix, publish as nanopublications
3. **Expert.AI claim extraction** — extract key claims from the dataset description

ROHub also auto-enriches with Expert.AI topic classification (layer 4), which happens automatically.

### Prerequisites

- `~/rohub_credentials.json`
- I-ADOPT service running locally (`docker compose up` in i-adopt-service-website)
- Internet access (for Expert.AI claim API and ROHub)

In [32]:
import rohub
import json
import requests
import fsspec
import datetime
import uuid
from pathlib import Path
from rohub import settings, utils

## Configuration

Set the RO-Crate to annotate. This notebook works with any of the three FAIR2Adapt examples.

In [30]:
# Choose one:
ro_pid = "https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056"  # RIOMAR
# ro_pid = "https://w3id.org/ro-id/fdc1c071-76d7-44df-a565-8217ebcc59fe"  # Sentinel-2
# ro_pid = "https://w3id.org/ro-id/1f0b5044-ae4f-483d-b7a2-48a5a6ac3965"  # NorESM2 ARCTIC

# Service endpoints
IADOPT_URL = "http://localhost:8000"  # I-ADOPT decomposition service
CLAIM_URL = "https://labdemos.expertcustomers.ai/services/claim_extraction"  # Expert.AI

# schema.org and I-ADOPT vocabulary
SCHEMA = "https://schema.org/"
DCTERMS = "http://purl.org/dc/terms/"
IADOPT = "https://w3id.org/iadopt/ont/"
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

ro_id = ro_pid.rstrip("/").split("/")[-1]
print(f"RO-Crate: {ro_pid}")

RO-Crate: https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056


## Login to ROHub and load the RO-Crate

In [11]:
credentials = json.loads(Path("~/rohub_credentials.json").expanduser().read_text())
rohub.login(username=credentials["username"], password=credentials["password"])

ro = rohub.ros_load(identifier=ro_id)
print(f"Title: {ro.title}")
print(f"Description: {ro.description[:200]}...")

resources = ro.list_resources()
print(f"\nResources:")
print(resources[["identifier", "type", "name", "url"]].to_string())

# Get the dataset URL
dataset = resources[resources["type"] == "Dataset"].iloc[0]
dataset_url = dataset["url"]
dataset_res = f"{ro_pid}/resources/{dataset['identifier']}"
print(f"\nDataset URL: {dataset_url}")

Logged successfully as annef@simula.no.
Research Object was successfully loaded with id = 24600867-23ca-4b97-be14-3aa63883c056
Title: FAIR2Adapt Processing Pipeline for Climate Change Adaptation
Description: High-resolution coastal ocean model outputs are essential for climate change adaptation strategies, yet their large size and complexity create significant barriers to accessibility and use by decision...

Resources:
                             identifier                  type                                          name                                                                                  url
0  c14c9055-e006-4b95-a24f-be8f788835b0           Web Service                                     Dashboard                                       https://fair2adapt.github.io/riomar-dashboard/
1  af29ccde-1f11-49b6-ab87-4efef4dff11d  Software source code  RIOMAR → HEALPix (DGGS) regridding notebooks                                      https://github.com/FAIR2Adapt/softwares_RIOMA

## Layer 1: Extract metadata from the data

Read CF attributes directly from the Zarr store to discover variables,
spatial/temporal coverage, and resolution. This is what the data creator
provided — the foundation for all other metadata layers.

In [12]:
# Read Zarr metadata (works with both public and proxy URLs)
zarr_url = dataset_url.rstrip("/")

# Try direct access, then with sublevels (for multiscale pyramids)
mapper = fsspec.get_mapper(zarr_url)
root_meta = json.loads(mapper["zarr.json"])
root_attrs = root_meta.get("attributes", {})

# Check for multiscale pyramid
multiscales = root_attrs.get("multiscales")
if multiscales:
    finest = multiscales["layout"][0]["asset"]
    var_meta = json.loads(mapper[f"{finest}/zarr.json"]).get("consolidated_metadata", {}).get("metadata", {})
    print(f"Multiscale pyramid detected, finest level: {finest}")
else:
    var_meta = root_meta.get("consolidated_metadata", {}).get("metadata", {})

print(f"\nRoot attributes: {list(root_attrs.keys())}")
print(f"Variables/arrays: {list(var_meta.keys())}")

Multiscale pyramid detected, finest level: 0

Root attributes: ['name', 'description', 'Conventions', 'title', 'rst_file', 'grd_file', 'ini_file', 'frc_file', 'VertCoordType', 'theta_s', 'theta_s_expl', 'theta_b', 'theta_b_expl', 'Tcline', 'Tcline_expl', 'Tcline_units', 'hc', 'hc_expl', 'hc_units', 'sc_w', 'Cs_w', 'sc_r', 'Cs_r', 'ndtfast', 'dt', 'dtfast', 'nwrt', 'tnu4_expl', 'units', 'rdrg', 'rdrg_expl', 'rdrg_units', 'rho0', 'rho0_expl', 'rho0_units', 'gamma2', 'gamma2_expl', 'x_sponge', 'v_sponge', 'sponge_expl', 'zarr_conventions', 'multiscales', 'dggs']
Variables/arrays: ['cell_ids', 'crs', 's_rho', 'salt', 'temp', 'time_counter', 'time_instant', 'zeta']


In [13]:
# Identify data variables (exclude coordinates and grid variables)
COORDINATE_NAMES = {
    "cell", "cell_ids", "crs", "time", "time_counter", "time_instant",
    "s_rho", "sigma", "depth", "depth_bnds", "latitude", "longitude",
    "lat", "lon", "bounds",
}

data_vars = {}
for vname, vmeta in var_meta.items():
    if vname in COORDINATE_NAMES:
        continue
    attrs = vmeta.get("attributes", {})
    if "long_name" in attrs or "standard_name" in attrs:
        data_vars[vname] = {
            "long_name": attrs.get("long_name", vname),
            "standard_name": attrs.get("standard_name"),
            "units": attrs.get("units", "dimensionless"),
            "shape": vmeta.get("shape", []),
        }

print("Data variables found:")
for vname, info in data_vars.items():
    print(f"  {vname}: {info['long_name']} [{info['units']}]")
    if info['standard_name']:
        print(f"    CF standard_name: {info['standard_name']}")

Data variables found:
  salt: salinity [PSU]
  temp: potential temperature [Celsius]
  zeta: free-surface [meter]


In [14]:
# Extract grid information
grid_info = {}

# Check for HEALPix
for vname in ["cell", "cell_ids", "crs"]:
    if vname in var_meta:
        attrs = var_meta[vname].get("attributes", {})
        if attrs.get("grid_name") == "healpix" or attrs.get("grid_mapping_name") == "healpix":
            level = attrs.get("level") or attrs.get("healpix_nside")
            nside = attrs.get("healpix_nside", 2 ** attrs.get("level", 0))
            grid_info = {
                "type": "HEALPix",
                "level": attrs.get("level"),
                "nside": nside,
                "indexing": attrs.get("indexing_scheme", attrs.get("healpix_order", "nested")),
            }
            break

# Check for curvilinear (2D lat/lon)
if not grid_info:
    for vname in ["latitude", "lat"]:
        if vname in var_meta and len(var_meta[vname].get("shape", [])) == 2:
            grid_info = {"type": "curvilinear"}
            break

if not grid_info:
    grid_info = {"type": "regular"}

# Resolution text
if grid_info["type"] == "HEALPix" and multiscales:
    levels = [json.loads(mapper[f"{l['asset']}/zarr.json"]).get("consolidated_metadata", {}).get("metadata", {}) 
              for l in [multiscales["layout"][0], multiscales["layout"][-1]]]
    finest_level = levels[0].get("cell", levels[0].get("cell_ids", {})).get("attributes", {}).get("level", "?")
    coarsest_level = levels[1].get("cell", levels[1].get("cell_ids", {})).get("attributes", {}).get("level", "?")
    resolution_text = f"HEALPix multiscale pyramid, level {finest_level} to level {coarsest_level}"
elif grid_info["type"] == "HEALPix":
    resolution_text = f"HEALPix level {grid_info['level']} (nside={grid_info['nside']})"
else:
    resolution_text = f"{grid_info['type']} grid"

print(f"Grid: {grid_info}")
print(f"Resolution: {resolution_text}")

Grid: {'type': 'HEALPix', 'level': None, 'nside': 8192, 'indexing': 'nested'}
Resolution: HEALPix multiscale pyramid, level ? to level ?


## Layer 2: I-ADOPT variable decomposition

For each data variable, we call the I-ADOPT service to decompose the
`long_name` into structured components (Property, ObjectOfInterest, Matrix)
and enrich with Wikidata URIs. The results can be published as nanopublications.

This step requires the I-ADOPT service running locally:
```bash
cd i-adopt-service-website && docker compose up
```

In [15]:
# Check if I-ADOPT service is available
try:
    r = requests.get(f"{IADOPT_URL}/health", timeout=5)
    iadopt_available = r.status_code == 200
except Exception:
    iadopt_available = False

print(f"I-ADOPT service: {'available' if iadopt_available else 'NOT RUNNING — skipping decomposition'}")

I-ADOPT service: available


In [16]:
decompositions = {}

if iadopt_available:
    for vname, info in data_vars.items():
        definition = f"{info['long_name']}"
        if info['units'] != 'dimensionless':
            definition += f" in {info['units']}"
        
        print(f"\nDecomposing: {definition}")
        
        resp = requests.post(
            f"{IADOPT_URL}/decompose",
            json={"definition": definition},
            timeout=300,
        )
        
        if resp.ok:
            result = resp.json()
            decompositions[vname] = result
            parsed = result.get("enriched_json") or result.get("parsed_json", {})
            print(f"  Property: {parsed.get('hasProperty', '?')}")
            print(f"  ObjectOfInterest: {parsed.get('hasObjectOfInterest', '?')}")
            print(f"  Matrix: {parsed.get('hasMatrix', '?')}")
            if result.get("ttl"):
                print(f"  TTL: {len(result['ttl'])} chars")
        else:
            print(f"  ERROR: {resp.status_code} {resp.text[:200]}")
else:
    print("Skipping I-ADOPT decomposition (service not running)")


Decomposing: salinity in PSU
  Property: salinity
  ObjectOfInterest: water
  Matrix: 
  TTL: 1393 chars

Decomposing: potential temperature in Celsius
  Property: temperature
  ObjectOfInterest: temperature
  Matrix: 
  TTL: 1771 chars

Decomposing: free-surface in meter
  Property: elevation
  ObjectOfInterest: free-surface
  Matrix: 
  TTL: 1585 chars


In [17]:
# Save decompositions locally for reuse
if decompositions:
    cache_file = Path(f"iadopt_cache_{ro_id[:8]}.json")
    cache_file.write_text(json.dumps(decompositions, indent=2))
    print(f"Saved {len(decompositions)} decompositions to {cache_file}")
    
    # Show TTL for the first variable
    first = list(decompositions.values())[0]
    if first.get("ttl"):
        print(f"\nExample TTL for '{list(decompositions.keys())[0]}':")
        print(first["ttl"][:500])

Saved 3 decompositions to iadopt_cache_24600867.json

Example TTL for 'salt':
@prefix iop: <https://w3id.org/iadopt/ont/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix pav: <http://purl.org/pav/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix orcid: <https://orcid.org/> .
@prefix fdof: <https://w3id.org/fdof/ontology#> .


<https://w3id.o


## Layer 3: Expert.AI claim extraction

Extract key claims from the dataset description using the Expert.AI service.
These claims capture the main statements about what the data shows or provides.

In [18]:
description = ro.description
print(f"Extracting claims from description ({len(description)} chars)...")
print(f"\n\"{description[:300]}...\"\n")

claims = []
try:
    resp = requests.post(
        CLAIM_URL,
        headers={"Content-Type": "text/plain"},
        data=description,
        timeout=30,
    )
    if resp.ok:
        raw_claims = resp.json()
        # Deduplicate
        seen = set()
        for c in raw_claims:
            text = c.get("claim", "").strip()
            if text and text.lower() not in seen:
                claims.append(text)
                seen.add(text.lower())
        print(f"Extracted {len(claims)} claims:")
        for i, c in enumerate(claims, 1):
            print(f"  {i}. {c}")
    else:
        print(f"Expert.AI returned {resp.status_code}")
except Exception as e:
    print(f"Expert.AI service error: {e}")

Extracting claims from description (1586 chars)...

"High-resolution coastal ocean model outputs are essential for climate change adaptation strategies, yet their large size and complexity create significant barriers to accessibility and use by decision-makers. The RiOMAR (River-influenced Ocean Margin) hindcast simulations (1980--2020) provide valuab..."

Extracted 9 claims:
  1. The RiOMAR hindcast simulations (1980-2020) reveal significant water quality degradation patterns along French Atlantic coasts, which can inform climate adaptation strategies.
  2. The FAIR data processing pipeline regrids curvilinear ocean model outputs to ellipsoidal HEALPix, converts to ARCO Zarr format, and enriches metadata using STAC with oceanographic domain semantics from I-ADOPT, resulting in FAIR Digital Objects.
  3. The FAIR2Adapt project's data processing pipeline successfully transforms RiOMAR ocean model outputs into FAIR Digital Objects by regridding curvilinear data to ellipsoidal HEALPix usi

## Add annotations to the RO-Crate

Now we combine all three layers into structured annotations on the RO-Crate,
following the pattern from `riomar_rocrate.ipynb`.

In [19]:
# Reload RO and create a new annotation
ro = rohub.ros_load(identifier=ro_id)
annot = ro.add_annotations()
aid = annot["identifier"]
print(f"Created annotation: {aid}")

# Product ID for this dataset
mvp_id = f"{ro_pid}/product/{uuid.uuid4().hex[:8]}"

Research Object was successfully loaded with id = 24600867-23ca-4b97-be14-3aa63883c056
Created annotation: a3feacb2-563d-4b23-bf61-4020214ce3ee


In [20]:
# Link product to RO-Crate
ro.add_triple(
    the_subject=ro_pid, the_predicate=f"{SCHEMA}isRelatedTo",
    the_object=mvp_id, annotation_id=aid, object_class="URIRef",
)
ro.add_triple(
    the_subject=mvp_id, the_predicate=RDF_TYPE,
    the_object=f"{SCHEMA}Product", annotation_id=aid, object_class="URIRef",
)

# Resolution
ro.add_triple(
    the_subject=mvp_id,
    the_predicate="http://data.europa.eu/930/spatialResolutionAsText",
    the_object=resolution_text, annotation_id=aid,
)

# Identifier
ro.add_triple(
    the_subject=mvp_id, the_predicate=f"{SCHEMA}identifier",
    the_object=dataset_url, annotation_id=aid,
)

# Provenance
ro.add_triple(
    the_subject=mvp_id, the_predicate=f"{DCTERMS}provenance",
    the_object="https://fair2adapt.github.io/riomar-dashboard/",
    annotation_id=aid, object_class="URIRef",
)

print(f"Product: {mvp_id}")
print(f"Resolution: {resolution_text}")

Product: https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056/product/e9743676
Resolution: HEALPix multiscale pyramid, level ? to level ?


In [21]:
# Add variables with I-ADOPT decomposition
for i, (vname, info) in enumerate(data_vars.items(), 1):
    var_id = f"{ro_pid}/variable/{i}"
    
    # Link variable to product
    ro.add_triple(
        the_subject=mvp_id, the_predicate=f"{SCHEMA}variableMeasured",
        the_object=var_id, annotation_id=aid, object_class="URIRef",
    )
    ro.add_triple(
        the_subject=var_id, the_predicate=RDF_TYPE,
        the_object=f"{IADOPT}Variable", annotation_id=aid, object_class="URIRef",
    )
    
    # Variable name
    var_label = info["long_name"]
    ro.add_triple(
        the_subject=var_id, the_predicate=f"{SCHEMA}name",
        the_object=var_label, annotation_id=aid,
    )
    
    # Add I-ADOPT decomposition if available
    decomp = decompositions.get(vname)
    if decomp:
        parsed = decomp.get("enriched_json") or decomp.get("parsed_json", {})
        
        # Property
        if parsed.get("hasProperty"):
            prop_id = f"{var_id}/property"
            ro.add_triple(
                the_subject=var_id, the_predicate=f"{IADOPT}hasProperty",
                the_object=prop_id, annotation_id=aid, object_class="URIRef",
            )
            ro.add_triple(
                the_subject=prop_id, the_predicate=f"{SCHEMA}name",
                the_object=parsed["hasProperty"], annotation_id=aid,
            )
            if parsed.get("hasPropertyURI"):
                ro.add_triple(
                    the_subject=prop_id, the_predicate=f"{SCHEMA}url",
                    the_object=parsed["hasPropertyURI"], annotation_id=aid, object_class="URIRef",
                )
        
        # ObjectOfInterest
        ooi = parsed.get("hasObjectOfInterest")
        if ooi and isinstance(ooi, str):
            ooi_id = f"{var_id}/entity"
            ro.add_triple(
                the_subject=var_id, the_predicate=f"{IADOPT}hasObjectOfInterest",
                the_object=ooi_id, annotation_id=aid, object_class="URIRef",
            )
            ro.add_triple(
                the_subject=ooi_id, the_predicate=f"{SCHEMA}name",
                the_object=ooi, annotation_id=aid,
            )
            if parsed.get("hasObjectOfInterestURI"):
                ro.add_triple(
                    the_subject=ooi_id, the_predicate=f"{SCHEMA}url",
                    the_object=parsed["hasObjectOfInterestURI"], annotation_id=aid, object_class="URIRef",
                )
        
        # Matrix
        matrix = parsed.get("hasMatrix")
        if matrix and isinstance(matrix, str):
            matrix_id = f"{var_id}/matrix"
            ro.add_triple(
                the_subject=var_id, the_predicate=f"{IADOPT}hasMatrix",
                the_object=matrix_id, annotation_id=aid, object_class="URIRef",
            )
            ro.add_triple(
                the_subject=matrix_id, the_predicate=f"{SCHEMA}name",
                the_object=matrix, annotation_id=aid,
            )
        
        # Link to TTL/nanopub if available
        if decomp.get("ttl"):
            ro.add_triple(
                the_subject=var_id, the_predicate=f"{DCTERMS}description",
                the_object=f"I-ADOPT decomposition: {parsed.get('hasProperty', '?')} of {ooi or '?'}",
                annotation_id=aid,
            )
    
    print(f"  {vname}: {var_label}")
    if decomp:
        parsed = decomp.get("enriched_json") or decomp.get("parsed_json", {})
        print(f"    I-ADOPT: {parsed.get('hasProperty', '?')} | {parsed.get('hasObjectOfInterest', '?')} | {parsed.get('hasMatrix', '?')}")

  salt: salinity
    I-ADOPT: salinity | water | 
  temp: potential temperature
    I-ADOPT: temperature | temperature | 
  zeta: free-surface
    I-ADOPT: elevation | free-surface | 


In [ ]:
# Add Expert.AI claims to RO-Crate
if claims:
    for i, claim_text in enumerate(claims, 1):
        claim_id = f"{ro_pid}/claim/{i}"
        ro.add_triple(
            the_subject=dataset_res, the_predicate=f"{SCHEMA}hasClaim",
            the_object=claim_id, annotation_id=aid, object_class="URIRef",
        )
        ro.add_triple(
            the_subject=claim_id, the_predicate=RDF_TYPE,
            the_object=f"{SCHEMA}Claim", annotation_id=aid, object_class="URIRef",
        )
        ro.add_triple(
            the_subject=claim_id, the_predicate=f"{SCHEMA}text",
            the_object=claim_text, annotation_id=aid,
        )
        print(f"  Claim {i}: {claim_text[:80]}..." if len(claim_text) > 80 else f"  Claim {i}: {claim_text}")
else:
    print("No claims to add")

## Publish claims as AIDA nanopublications

Convert the extracted claims into AIDA (Atomic, Independent, Declarative, Absolute) sentence
nanopublications using [ScienceLive](https://sciencelive4all.org/).

Requires a nanopub signing profile at `~/Documents/ScienceLive/ai-profile/profile.yml`
(software agent with keys). See `create_aida_nanopub.ipynb` for profile setup.

In [35]:
import urllib.parse
from datetime import datetime, timezone
from rdflib import Dataset as RDFDataset, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from nanopub import Nanopub, NanopubConf, load_profile

# ScienceLive configuration
PROFILE_PATH = Path("~/Documents/ScienceLive/ai-profile/profile.yml").expanduser()
PUBLISHER = "https://sciencelive4all.org/"
TEMP_NP = Namespace("https://w3id.org/sciencelive/np")

# Nanopub namespaces
NP_NS = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
HYCL = Namespace("http://purl.org/petapico/o/hycl#")
CITO = Namespace("http://purl.org/spar/cito/")
SCHEMA_NS = Namespace("http://schema.org/")

# AIDA template URIs
AIDA_TEMPLATE = URIRef("https://w3id.org/np/RALmXhDw3rHcMveTgbv8VtWxijUHwnSqhCmtJFIPKWVaA")
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RA0J4vUn_dekg-U1kK3AOEt02p9mT2WO03uGxLDec1jLw")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")

HYCL_AIDA_SENTENCE = URIRef("http://purl.org/petapico/o/hycl#AIDA-Sentence")
HYCL_NS_URI = URIRef("http://purl.org/petapico/o/hycl")

profile = load_profile(str(PROFILE_PATH))
print(f"Profile: {profile.name} ({profile.orcid_id})")

Profile: claude-ai-agent (https://w3id.org/np/RAIA9ECaN2ypOVvl4YeNjT6nbpwko9xMcctxB_uYscLG4/claude-ai-agent)


In [36]:
def create_aida_nanopub(aida_sentence, related_dataset, author_uri, author_name):
    """Create an AIDA sentence nanopublication as an rdflib Dataset."""
    this_np = URIRef(TEMP_NP)
    head_graph = URIRef(TEMP_NP + "/Head")
    assertion_graph = URIRef(TEMP_NP + "/assertion")
    provenance_graph = URIRef(TEMP_NP + "/provenance")
    pubinfo_graph = URIRef(TEMP_NP + "/pubinfo")

    author = URIRef(author_uri) if author_uri.startswith("http") else URIRef(f"https://orcid.org/{author_uri}")
    aida_uri = URIRef(f"http://purl.org/aida/{urllib.parse.quote(aida_sentence, safe='')}")

    ds = RDFDataset()
    ds.bind("this", TEMP_NP)
    ds.bind("sub", TEMP_NP)
    ds.bind("np", NP_NS)
    ds.bind("dct", DCT)
    ds.bind("nt", NT)
    ds.bind("npx", NPX)
    ds.bind("xsd", XSD)
    ds.bind("rdfs", RDFS)
    ds.bind("prov", PROV)
    ds.bind("foaf", FOAF)
    ds.bind("hycl", HYCL)
    ds.bind("cito", CITO)
    ds.bind("schema", SCHEMA_NS)

    # HEAD
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP_NS.Nanopublication))
    head.add((this_np, NP_NS.hasAssertion, assertion_graph))
    head.add((this_np, NP_NS.hasProvenance, provenance_graph))
    head.add((this_np, NP_NS.hasPublicationInfo, pubinfo_graph))

    # ASSERTION
    assertion = ds.graph(assertion_graph)
    assertion.add((aida_uri, RDF.type, HYCL_AIDA_SENTENCE))
    assertion.add((aida_uri, CITO.obtainsSupportFrom, URIRef(related_dataset)))

    # PROVENANCE
    provenance = ds.graph(provenance_graph)
    provenance.add((assertion_graph, PROV.wasAttributedTo, author))

    # PUBINFO
    pubinfo = ds.graph(pubinfo_graph)
    pubinfo.add((author, FOAF.name, Literal(author_name)))
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef(PUBLISHER)))
    pubinfo.add((this_np, NPX.hasNanopubType, HYCL_AIDA_SENTENCE))
    pubinfo.add((this_np, NPX.hasNanopubType, HYCL_NS_URI))
    pubinfo.add((this_np, NPX.introduces, aida_uri))
    label = f"AIDA sentence: {aida_sentence[:100]}..." if len(aida_sentence) > 100 else f"AIDA sentence: {aida_sentence}"
    pubinfo.add((this_np, RDFS.label, Literal(label)))
    pubinfo.add((this_np, NT.wasCreatedFromTemplate, AIDA_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))

    return ds, label

print("AIDA nanopub function defined")

AIDA nanopub function defined


In [37]:
#claims = [
#      "The FAIR2Adapt project's data processing pipeline successfully transforms RiOMAR ocean model outputs into FAIR Digital Objects by regridding curvilinear data to ellipsoidal HEALPix using the healpix-geo library.",
#      "The RiOMAR hindcast simulations (1980-2020) reveal significant water quality degradation patterns along French Atlantic coasts, which can inform climate adaptation strategies.",
#      "The FAIR data processing pipeline regrids curvilinear ocean model outputs to ellipsoidal HEALPix, converts to ARCO Zarr format, and enriches metadata using STAC with oceanographic domain semantics from I-ADOPT, resulting in FAIR Digital Objects.",
#      "The FAIR2Adapt project's data processing pipeline transforms RiOMAR outputs into Analysis-Ready Cloud-Optimized (ARCO) Zarr format.",
#      "The pipeline incorporates STAC (SpatioTemporal Asset Catalog) with oceanographic domain semantics from I-ADOPT to enhance metadata enrichment.",
#      "The FAIR2Adapt project's data processing pipeline adheres to AGILE Reproducible Paper Guidelines, utilizing version-controlled workflows, containerized environments, and comprehensive provenance metadata to ensure reproducibility.",
#      "The FAIR data processing pipeline transforms RiOMAR outputs into FAIR Digital Objects, achieving over 90% compliance in pilot demonstrations on representative datasets.",
#      "The FAIR2Adapt pipeline transforms RiOMAR hindcast simulations into FAIR Digital Objects, making high-resolution coastal ocean model outputs accessible and analyzable for climate adaptation planning.",
#      "The FAIR Digital Objects are packaged in RO-Crates with nanopublications and registered in ROHub for discovery and reuse, enabling the sharing and reuse of data.",
#  ]
#dataset_url = "https://pangeo-eosc-minioapi.vm.fedcloud.eu/afouilloux-riomar/small_hp_pyramid.zarr"
#ro_id = "24600867-23ca-4b97-be14-3aa63883c056"

In [ ]:
# Generate, sign, and publish AIDA nanopubs for each claim
PUBLISH = False  # Set to True when ready to publish to the nanopub network

output_dir = Path(f"outputs/nanopubs_{ro_id[:8]}")
output_dir.mkdir(parents=True, exist_ok=True)

published_uris = []

if claims:
    for i, claim_text in enumerate(claims, 1):
        ds_rdf, label = create_aida_nanopub(
            aida_sentence=claim_text,
            related_dataset=dataset_url,
            author_uri=profile.orcid_id,
            author_name=profile.name,
        )

        # Save TriG file
        trig_file = output_dir / f"claim_{i}.trig"
        ds_rdf.serialize(destination=str(trig_file), format="trig")
        print(f"  {i}. {trig_file.name}: {claim_text[:70]}...")

        if PUBLISH:
            conf = NanopubConf(profile=profile)
            np_obj = Nanopub(rdf=trig_file, conf=conf)
            np_obj.sign()
            signed_path = trig_file.with_suffix(".signed.trig")
            np_obj.store(signed_path)
            np_obj.publish()
            published_uris.append(np_obj.source_uri)
            print(f"     Published: {np_obj.source_uri}")

    print(f"\nGenerated {len(claims)} nanopubs in {output_dir}/")
    if PUBLISH:
        print(f"Published {len(published_uris)} nanopubs")
    else:
        print("Set PUBLISH = True to sign and publish to the nanopub network")
else:
    print("No claims to publish")

# Add published nanopubs as resources in the RO-Crate output folder
if published_uris:
    ro = rohub.ros_load(identifier=ro_id)
    folders = ro.list_folders()
    output_folder = folders[folders["name"] == "output"]
    output_folder_path = output_folder.iloc[0]["path"] if not output_folder.empty else None

    # Also link nanopub URIs to claims in annotations
    link_annot = ro.add_annotations()
    link_aid = link_annot["identifier"]

    for i, (np_uri, claim_text) in enumerate(zip(published_uris, claims), 1):
        # Add as resource in output folder
        short_title = claim_text[:80] + "..." if len(claim_text) > 80 else claim_text
        ro.add_external_resource(
            input_url=np_uri,
            res_type="Bibliographic Resource",
            title=f"AIDA Claim {i}: {short_title}",
            folder=output_folder_path,
        )

        # Link claim to nanopub in annotations
        claim_id = f"{ro_pid}/claim/{i}"
        ro.add_triple(
            the_subject=claim_id,
            the_predicate="http://purl.org/dc/terms/isReferencedBy",
            the_object=np_uri,
            annotation_id=link_aid,
            object_class="URIRef",
        )
        print(f"  Added resource + link: Claim {i} → {np_uri.split('/')[-1][:20]}...")

    print(f"\n{len(published_uris)} nanopubs added to RO-Crate output folder")

## Verify

In [23]:
print(f"RO-Crate: {ro_pid}")
print(f"Annotation: {aid}")
print(f"\nSummary:")
print(f"  Variables: {len(data_vars)}")
print(f"  I-ADOPT decompositions: {len(decompositions)}")
print(f"  Expert.AI claims: {len(claims)}")
print(f"  Grid: {resolution_text}")
print(f"\nView in ROHub: https://reliance.rohub.org/explore/{ro_id}")
print(f"Test with FDO2map: python fdo2map.py {ro_pid}")

RO-Crate: https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056
Annotation: a3feacb2-563d-4b23-bf61-4020214ce3ee

Summary:
  Variables: 3
  I-ADOPT decompositions: 3
  Expert.AI claims: 9
  Grid: HEALPix multiscale pyramid, level ? to level ?

View in ROHub: https://reliance.rohub.org/explore/24600867-23ca-4b97-be14-3aa63883c056
Test with FDO2map: python fdo2map.py https://w3id.org/ro-id/24600867-23ca-4b97-be14-3aa63883c056
